# Quantization Troubleshooting with the Model Compression Toolkit (MCT) Using the XQuant Extension Tool

[Run this tutorial in Google Colab](https://colab.research.google.com/github/SonySemiconductorSolutions/mct-model-optimization/blob/main/tutorials/notebooks/mct_features_notebooks/pytorch/example_pytorch_XQuant_Extension_Tool.ipynb)

## Overview
This notebook provides practical guidance for improving the quality of post‑training quantization for PyTorch models using the XQuant Extension Tool. 
It computes the error for each layer by comparing the floating‑point model and the quantized model, and combines these results with the quantization log. The analysis is presented as a report that highlights the causes of detected errors and suggests appropriate corrective actions for each.

## Summary
We will cover the following steps:

1. Load a pre-trained MobileNetV3 model.
2. perform Post-Training Quantization using MCT (no correction).
3. Use the XQuant Extension Tool
   - Understanding the Quantization Error Graph
   - Quantization Troubleshooting for MCT  
4. Judgeable Troubleshoot
   - Outlier Removal
5. General Troubleshoot
   - Representative Dataset Size & Diversity
   - Bias Correction
   - Using More Samples in Mixed Precision Quantization
   - Threshold Selection Error Method
   - Enabling Hessian Based Mixed Precision
   - GPTQ - Gradient-Based Post Training Quantization
6. Conclusion

    
## Setup
Install the relevant packages:

In [ ]:
# !pip install torch==2.6.0 torchvision==0.21.0

## Define a Random Data Generator
For demonstration purposes, we will use a random dataset generator to create both the representative dataset and the validation dataset. This will allow us to simulate data for quantization and validation without using an actual dataset.

In [ ]:
# Function to generate random data.

## perform Post-Training Quantization using MCT (no correction)

In [ ]:
# post-training quantization 

The accuracy was ____. We will improve this.

## Use the XQuant Extension Tool

In [ ]:
# xquant_report_troubleshoot_pytorch_experimental

## Judgeable Troubleshoot
### Understanding the Quantization Error Graph

Six quantization error graphs will be generated in the directory specified by report_dir in XQuantConfig.

These graphs display three metrics — MSE, cosine similarity, and SQNR — for two datasets: the representative dataset and the validation dataset.

The quantization error indicates the difference in each layer’s output between the floating-point model and the quantized model.

Each layer’s quantization error is compared with threshold_quantize_error to identify layers that exhibit significant behavioral changes after quantization.

As an example, the figure below shows an output graph computed using the “mse” metric with the representative dataset. A default threshold value of 0.1 is set, and layers exceeding this threshold are marked with red circles. In addition, the corresponding layer names on the X-axis are highlighted in red.




Figure: quant_loss_mse_repr.png


- X-axis: Layer names (layers identified as degraded are highlighted in red)
- Y-axis: Quantization error
- Red dashed line: Threshold for accuracy degradation, as defined in XQuantConfig
- Red circle: Layers judged to have degraded accuracy


From the graph, we can see that the layer XX is deteriorating.

### Quantization Troubleshooting for MCT

The Model Compression Toolkit (MCT) offers numerous functionalities to compress neural networks with minimal accuracy loss. However, in some cases, the compressed model may experience a significant decrease in accuracy.

This lost accuracy can often be recovered by adjusting the quantization configuration or setup.
Listed below are a series of steps designed to help you recover any accuracy lost during compression with MCT. Some steps may apply to your model, while others may not.

The XQuant Extension Tool automatically detects potential issues and displays the relevant warning messages in the console. Please refer to the corresponding troubleshooting guide and modify the settings as needed.

See [TroubleShooting Manual](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/index.html)


### Outlier Removal
The quantization accuracy may degrade when there are outlier activations in the quantized layers of your model.

You can check if there are any outliers in your activation tensor by visualizing the histogram (shown below) generated in the directory specified by report_dir in XQuantConfig.

This graph shows that an outlier occurs near XX, so we will correct z_threshold to that value below.



See [TroubleShooting Documentation>>Outlier Removal](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/outlier_removal.html#ug-outlier-removal)

In [ ]:
# Set z_threshold to 1.0 in the XQuant configuration and re-run the quantization process.

## General Troubleshoots
If there is no significant improvement, comprehensively evaluate other areas for improvement.
The following items are general troubleshoots for quantization accuracy improvement.

### Representative Dataset Size & Diversity
The representative dataset is used by MCT to derive the threshold for the model's activation tensor.
If the representative dataset is too small or not diverse enough, accuracy may decrease.
Increase the number of samples in the representative dataset or increase the diversity of the samples.

See [TroubleShooting Documentation>>Representative Dataset Size & Diversity](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/representative_dataset_size_and_diversity.html#ug-representative-dataset-size-and-diversity)

In [ ]:
# Change representative dataset and re-run the quantization process.

### Bias Correction
MCT applies bias correction by default to overcome induced bias shift caused by weights quantization.

You can check if the bias correction causes a degradation in accuracy, by disabling the bias correction (setting weights_bias_correction to False of the QuantizationConfig in CoreConfig).

See [TroubleShooting Documentation>>Bias Correction](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/bias_correction.html#ug-bias-correction)

In [ ]:
# Change weights_bias_correction and re-run the quantization process.

### Using More Samples in Mixed Precision Quantization

In Mixed Precision quantization, MCT will assign a different bit width to each weight in the model, depending on the weight’s layer sensitivity and a resource constraint defined by the user, such as target model size.

By default, MCT employs 32 samples from the provided representative dataset for the Mixed Precision search. Leveraging a larger dataset could enhance results, particularly when dealing with datasets exhibiting high variance.

Set the num_of_images attribute to a larger value of the MixedPrecisionQuantizationConfig in CoreConfig.

See [TroubleShooting Documentation>>Using More Samples in Mixed Precision Quantization](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/using_more_samples_in_mixed_precision_quantization.html#ug-using-more-samples-in-mixed-precision-quantization

In [ ]:
# Change num_of_images and re-run the quantization process.

### Threshold Selection Error Method
MCT defaults to employing the Mean-Squared Error (MSE) metric for threshold optimization, however, it offers a range of alternative error metrics (e.g. using min/max values, KL-divergence, etc.) to accommodate different network requirements.

We advise you to consider other error metrics if your model is suffering from significant accuracy degradation, especially if it contains unorthodox activation layers.

For example, set NOCLIPPING to the activation_error_method attribute of the QuantizationConfig in CoreConfig.

See [TroubleShooting Documentation>>Threshold Selection Error Method](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/threhold_selection_error_method.html#ug-threshold-selection-error-method)

In [ ]:
# Change activation_error_method and re-run the quantization process.

### Enabling Hessian Based Mixed Precision
MCT offers a Hessian-based scoring mechanism to assess the importance of layers during the Mixed Precision search.
This feature can notably enhance Mixed Precision outcomes for certain network architectures.

Set the use_hessian_based_scores flag to True in the MixedPrecisionQuantizationConfig of the CoreConfig.

See [TroubleShooting Documentation>>Enabling Hessian-Based Mixed Precision](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/enabling_hessian-based_mixed_precision.html#ug-enabling-hessian-based-mixed-precision)


In [ ]:
# Change use_hessian_based_scores and re-run the quantization process.

### GPTQ - Gradient-Based Post Training Quantization
When PTQ (either with or without Mixed Precision) fails to deliver the required accuracy, GPTQ is potentially the remedy.

MCT can configure GPTQ optimization options, such as the number of epochs for the optimization process.

See [GPTQ - Gradient-Based Post Training Quantization](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/gptq-gradient_based_post_training_quantization.html#ug-gptq-gradient-based-post-training-quantization)

In [ ]:
# Re-run the quantization process using GPTQ.

## Conclusion
Through this XQuant analysis, accuracy improved by XX%

## Copyrights
Copyright 2026 Sony Semiconductor Solutions, Inc. All rights reserved.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
